In [1]:
from typing import TypedDict

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.messages import ToolMessage
from langchain.agents.middleware import wrap_model_call, wrap_tool_call, ModelRequest, ModelResponse, dynamic_prompt

In [2]:
basic_model = ChatOpenAI(
    model="Qwen3.5-0.8B-MLX-4bit",
    base_url="http://localhost:8000/v1",
    api_key="omlx-12345678",
    temperature=0.1,
    max_tokens=1000,
    timeout=120
)
advanced_model = ChatOpenAI(
    model="Qwen3.5-2B-MLX-4bit",
    base_url="http://localhost:8000/v1",
    api_key="omlx-12345678",
    temperature=0.1,
    max_tokens=1000,
    timeout=120
)

In [8]:
@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

In [9]:
@wrap_tool_call
def handle_tool_errors(request, handler):
    """使用自定义消息处理工具执行错误。"""
    try:
        return handler(request)
    except Exception as e:
        # 向模型返回自定义错误消息
        return ToolMessage(
            content=f"工具错误：请检查您的输入并重试。({str(e)})",
            tool_call_id=request.tool_call["id"]
        )

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """根据对话复杂性选择模型。"""
    message_count = len(request.state["messages"])

    if message_count > 10:
        # 对较长的对话使用高级模型
        model = advanced_model
    else:
        model = basic_model

    request.model = model
    return handler(request)

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """根据用户角色生成系统提示。"""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "你是一个有帮助的助手。"

    if user_role == "expert":
        return f"{base_prompt} 提供详细的技术响应。"
    elif user_role == "beginner":
        return f"{base_prompt} 简单解释概念，避免使用行话。"

    return base_prompt

In [4]:
agent = create_agent(
    model=basic_model,  # 默认模型
    tools=[get_weather],
    middleware=[dynamic_model_selection, handle_tool_errors, user_role_prompt],
    system_prompt="你是一个有帮助的助手。请简洁准确。"
)

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "解释机器学习"}]},
    context={"user_role": "expert"}
)
print(result)

## 结构化输出

In [3]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, ProviderStrategy



class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

agent = create_agent(
    model=basic_model,
    # tools=[search_tool],
    # response_format=ToolStrategy(ContactInfo)
    response_format=ProviderStrategy(ContactInfo) # 模型提供商的原生结构化输出
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "从以下内容提取联系信息：John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')